In [21]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import sys

# dev = "laptop"
# title = f"UR10e IK benchmark - Laptop"

dev = "cluster"
title = f"UR10e IK benchmark - Cluster"

fname = f"benchmark_results_{dev}.json"

with open(fname) as f:
    raw = json.load(f)

data = {}
for solver, ops in raw.items():
    data[solver] = {}
    for op, vals in ops.items():
        data[solver][op] = {int(k): v for k, v in vals.items()}

SOLVERS = [
    ("jax_cpu",       "JAX CPU",      "#378ADD", "-",  "o"),
    ("jax_gpu",       "JAX GPU",      "#1D9E75", "-",  "s"),
    ("numba_serial",  "Numba serial CPU", "#BA7517", "-",  "^"),
    ("numba_prange",  "Numba prange CPU", "#D85A30", "--", "D"),
]
SCALE_SOLVERS = {"jax_cpu", "jax_gpu", "numba_serial"}

for op in ("fk", "ik"):
    fig, ax = plt.subplots(figsize=(7, 5))
    lines, labels, scale_vals = [], [], []

    for solver_key, label, color, ls, marker in SOLVERS:
        if solver_key not in data:
            continue
        op_data = data[solver_key].get(op, {})
        if not op_data:
            continue
        Bs   = sorted(op_data)
        mus  = [op_data[B] for B in Bs]
        mcps = [1 / v for v in mus]   # M calls/s  (1 / µs = 1e6 calls/s = 1 Mcall/s)
        l, = ax.plot(Bs, mcps, color=color, linestyle=ls,
                     marker=marker, markersize=5, linewidth=1.8, label=label)
        lines.append(l); labels.append(label)
        if solver_key in SCALE_SOLVERS:
            scale_vals.extend(mcps)

    # ── left axis : M calls/s ─────────────────────────────────────────────────
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("Batch size", fontsize=11)
    ax.set_ylabel("Throughput (M calls/s)", fontsize=11)
    ax.set_title(title, fontsize=12, fontweight="500")
    ax.grid(True, which="both", linestyle=":", linewidth=0.5, alpha=0.6)
    ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{x:g}"))
    ax.yaxis.set_minor_formatter(ticker.NullFormatter())

    if scale_vals:
        ax.set_ylim(min(scale_vals) * 0.5, max(scale_vals) * 2.0)

    ax.legend(lines, labels, fontsize=9, loc="upper left",
              framealpha=0.85, edgecolor="#cccccc")
    plt.tight_layout()
    for ext in ("png", "pdf"):
        out = f"benchmark_{dev}_{op}.{ext}"
        plt.savefig(out, bbox_inches="tight", dpi=150)
        print(f"Saved {out}")
    plt.close()

Saved benchmark_cluster_fk.png
Saved benchmark_cluster_fk.pdf
Saved benchmark_cluster_ik.png
Saved benchmark_cluster_ik.pdf
